# Phase 1: foundations

Your tasks: understand the files, implement the scorer, inspect labeled groups, and design validation.

Setup and safe previews are provided. Matching and modeling are yours to implement. Start with **Amazon ML (local)** as the kernel. Run cells in order. Exercise checks are skipped until you enable them; skipped does not mean passed.

In [1]:
from pathlib import Path
import sys, os

# Find the project from either its root or notebooks folder.
cwd = Path.cwd().resolve()
ROOT = next((p for p in [cwd, *cwd.parents] if (p / 'project_config.json').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from inside the project folder.')
sys.path.insert(0, str(ROOT / 'src'))
for name in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ.setdefault(name, '2')
from business_entity_resolution.workspace import CONFIG, DATASET_DIR, dataset_files, iter_tsv, preview
print('Project:', ROOT)
print('Python:', sys.executable)
print('Seed:', CONFIG['seed'])


Project: /home/a/Projects/Amazon ML hackathon
Python: /home/a/Projects/Amazon ML hackathon/.venv/bin/python
Seed: 2026


## 1. Check available resources
Installed RAM and currently available RAM are different. Keep previews small. The configured 4 GiB target is guidance, not a memory limiter.

In [2]:
import psutil
memory = psutil.virtual_memory()
print(f'Total RAM: {memory.total / 2**30:.1f} GiB; available: {memory.available / 2**30:.1f} GiB')
for path in dataset_files():
    print(path.name, 'OK' if path.is_file() else 'MISSING')


Total RAM: 15.3 GiB; available: 8.6 GiB
train_source1.tsv OK
train_source2.tsv OK
train_source3.tsv OK
test_source1.tsv OK
test_source2.tsv OK
test_source3.tsv OK
train_ground_truth.tsv OK


## 2. Inspect small previews
These are only file heads, not random samples or a valid training subset. Empty fields remain empty strings. Only five rows per file are materialized.

In [3]:
import pandas as pd
from IPython.display import display
for path in dataset_files():
    print(path.name)
    display(pd.DataFrame(preview(path, limit=CONFIG['preview_rows'])))


train_source1.tsv
train_source2.tsv
train_source3.tsv
test_source1.tsv
test_source2.tsv
test_source3.tsv
train_ground_truth.tsv


,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India


,entity_id,business_name,business_address,country
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India
3,S2-163963287,Summit Inc,"GREENSBORO, NC, 19 1/2 STARDUST TRAIL",US
4,S2-49942811,Delta Tetlecommunication Inc,"914 PIERPONT AVE, CLEVELAND, OH",US


,entity_id,business_name,business_address,country
0,S3-202863386,wilfordhancock.com,"Mack Rd, Haltom City, Texas",US
1,S3-859268022,International South Consultants Private Ltd,,India
2,S3-22467283,LLC Moncada Léarning Center,"5780 Fawn Ct, Fort Worth, Texas",US
3,S3-671162755,Moyna's Coffee,"1 Ivanhoe Ave, PO Box 6009, Cincinnati, Ohio",US
4,S3-960981775,Pvt. EFS Print Ventures Ltd.,"Door No 183, 41St Cross, 22Nd Main 9Th Block J...",India


,entity_id,business_name,business_address,country
0,S1-714132312,Zephay Labs Inc,"2621 Cotten Road, Tyler, TX",US
1,S1-106407869,Vision Partners Corp,"IA, Iowa City, 1064 Newton Rd, Unit 11",US
2,S1-156285671,<< Team Ecole,"175 Boulevard du Président Franklin Roosevelt,...",France
3,S1-689823050,Red Perfect Trading,"Mirzapur, Ews 12, Uttar Pradesh, Mirzapursadar...",India
4,S1-921369899,ZNB Club SARL,"Nouvelle-Aquitaine, La Teste-de-Buch, 5 bis Ru...",France


,entity_id,business_name,business_address,country
0,S2-192345572,Brahma Infosoft,"COIMATORE COLONY, HUNSUR TQMYSORE DIST., Karna...",India
1,S2-566025912,Marina Ecole France Sarl,"63 R. DE DIEPPE, LILLE, Hauts-de-France",France
2,S2-158121477,SCI Ptit Àmicale,"18 RUE JEN ZAY, Dunkerque, Nord",France
3,S2-89663826,Apex Summit,"67 KENTUCKY ST, SALYERSVILLE, KY",US
4,S2-884102769,Fresh Truist,"8264 FILLY COURT, ROANOKE COUNTY, VA",US


,entity_id,business_name,business_address,country
0,S3-462677478,मॉडर्न फाइनेंस,"No 10 Enkay Square, 448A, Udyog Vihar Phase V,...",India
1,S3-374810425,Shri Sai Infratech Co,"3/115, East Delhi, DL",India
2,S3-198586129,Fractales Amis Groupe S.A.S,"23 Rue Icmre, La Teste-de-buch, Gironde",France
3,S3-10300249,Shri Supreme Consulting Private (Limited),"H.no 910 A 3503, Mumbai, महाराष्ट्र",India
4,S3-604980231,Prime Realty Ventures Public Limited,"G.t. Karnal Road, Industrial Area, New Delhi, ...",India


,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."


**Your notes:**
- Why are these files tab-separated?
- Why might a business have multiple matching Source 2 records?
- How is an absent address different from a conflicting address?

Write your answers below.

**Our notes:**
- TSV is required because addresses and comma-separated ID lists can contain commas; tabs separate fields unambiguously.
- Multiple Source 2/3 rows can represent the same business because sources contain duplicate, historical, transliterated, or differently formatted variants.
- An absent address is missing evidence; a conflicting address is negative evidence. They must not be treated as the same condition.


## 3. Read the structural audit
The setup audit is a convenience. It does not verify cross-file ID integrity or business identity. Do not run an expensive full scan every time you restart the notebook.

In [4]:
import json
report_path = ROOT / 'reports' / 'data_audit_full.json'
if not report_path.exists():
    report_path = ROOT / 'reports' / 'data_audit_preview.json'
if report_path.exists():
    audit = json.loads(report_path.read_text(encoding='utf-8'))
    print('Audit scope:', audit['scope'])
    display(pd.DataFrame([{'file': Path(r['file']).name, 'rows_scanned': r['rows_scanned']} for r in audit['files']]))
    print('Limitations:', audit['limitations'])
else:
    print('Run scripts/audit_data.py from a terminal first.')


Audit scope: full
Limitations: ['No global ID uniqueness or cross-file referential-integrity checks.', 'Preview rows are file heads, not a random or representative sample.', 'Counts do not establish label correctness or a modeling baseline.']


,file,rows_scanned
0,train_source1.tsv,2206821
1,train_source2.tsv,5034616
2,train_source3.tsv,5285603
3,test_source1.tsv,1732544
4,test_source2.tsv,4887273
5,test_source3.tsv,5082316
6,train_ground_truth.tsv,2206821


## 4. Your ground-truth parser
Implement a function that converts a blank field to an empty set, and a nonblank field to matching IDs. Check duplicate IDs before set conversion. Decide how malformed IDs should be reported.

In [5]:
import re


def parse_match_ids(text):
    """Parse a truth/prediction ID list without hiding malformed labels."""
    if not isinstance(text, str):
        raise TypeError('matched IDs must be a string')
    if not text.strip():
        return set()

    ids = [value.strip() for value in text.split(',')]
    if any(not value for value in ids):
        raise ValueError(f'empty entity ID in {text!r}')
    if any(not re.fullmatch(r'S[23]-[^,\s]+', value) for value in ids):
        raise ValueError(f'malformed target ID in {text!r}')
    if len(ids) != len(set(ids)):
        raise ValueError(f'duplicate target ID in {text!r}')
    return set(ids)


assert parse_match_ids('') == set()
assert parse_match_ids('S2-123,S3-456') == {'S2-123', 'S3-456'}
for invalid_text in ('S2-123,S2-123', 'S1-123', 'S2-123,', 'S2-123, bad',
                     'S2-12 3', 'S2-12\t3', 'S2-12\n3'):
    try:
        parse_match_ids(invalid_text)
    except ValueError:
        pass
    else:
        raise AssertionError(f'Expected malformed input to fail: {invalid_text!r}')
print('Parser checks pass.')


Parser checks pass.


## 5. Your entity scorer
Let T be true IDs and P be predicted IDs. Derive TP, FP, FN using sets.

F0.5 = 1.25 TP / (1.25 TP + FP + 0.25 FN).

When both sets are empty, score 1. The official macro score averages over every required Source 1 entity, not classes or pairs.

In [6]:
def score_entity(true_ids, predicted_ids):
    """Return the official precision-weighted F0.5 for one reference."""
    true_ids = set(true_ids)
    predicted_ids = set(predicted_ids)
    true_positive = len(true_ids & predicted_ids)
    false_positive = len(predicted_ids - true_ids)
    false_negative = len(true_ids - predicted_ids)
    denominator = 1.25 * true_positive + false_positive + 0.25 * false_negative
    return 1.0 if denominator == 0 else 1.25 * true_positive / denominator


In [7]:
from math import isclose
run_score_checks = True  # Set to False only while editing the scorer.
cases = [
    ({'A', 'B'}, {'A', 'B'}, 1.0),
    ({'A', 'B'}, {'A'}, 5 / 6),
    ({'A', 'B'}, {'A', 'B', 'C'}, 5 / 7),
    ({'A', 'B'}, set(), 0.0),
    (set(), set(), 1.0),
    (set(), {'C'}, 0.0),
]
if run_score_checks:
    for truth, prediction, expected in cases:
        actual = score_entity(truth, prediction)
        assert isclose(actual, expected, rel_tol=1e-9, abs_tol=1e-12), (truth, prediction, actual, expected)
    print('All six entity-score examples pass.')
else:
    print('Exercise checks SKIPPED. Implement the function, then enable checks.')


All six entity-score examples pass.


**Next exercise:** implement macro averaging with a check that every required reference has a prediction row. The last three examples together must score 1/3. Predicting empty for all training references must score 123247 / 2206821, approximately 0.05585. Explain why pair accuracy would be misleading.

In [8]:
from statistics import mean


def macro_score(required_ids, truth_by_reference, predictions_by_reference):
    """Average each required reference once; reject invalid evaluation inputs.

    Prediction mappings must have exactly the required keys. Truth may contain
    additional references so one full truth mapping can serve a held-out subset.
    Detect duplicate prediction rows before constructing the mapping upstream.
    """
    required_ids = list(required_ids)
    required_set = set(required_ids)
    if not required_ids:
        raise ValueError('required reference list must not be empty')
    if len(required_ids) != len(required_set):
        raise ValueError('duplicate required reference IDs would change macro weighting')
    missing = required_set - predictions_by_reference.keys()
    extra = predictions_by_reference.keys() - required_set
    if missing or extra:
        raise KeyError(f'prediction coverage mismatch: {len(missing)} missing, {len(extra)} extra')
    missing_truth = required_set - truth_by_reference.keys()
    if missing_truth:
        raise KeyError(f'missing truth for {len(missing_truth)} required references')
    return mean(
        score_entity(truth_by_reference[reference_id], predictions_by_reference[reference_id])
        for reference_id in required_ids
    )

# The three entities below score (1 + 0 + 0) / 3 = 1/3.
example_truth = {
    'S1-A': {'S2-A'},
    'S1-B': {'S2-B'},
    'S1-C': set(),
}
example_predictions = {
    'S1-A': {'S2-A'},
    'S1-B': set(),
    'S1-C': {'S2-C'},
}
assert isclose(macro_score(example_truth, example_truth, example_predictions), 1 / 3)
# Check both mathematical aggregation and input-contract failures.
assert isclose(macro_score(['S1-A', 'S1-B'], example_truth,
                           {'S1-A': {'S2-A'}, 'S1-B': set()}), 0.5)
invalid_macro_cases = [
    ('missing prediction', KeyError, ['S1-A', 'S1-B'], example_truth,
     {'S1-A': {'S2-A'}}),
    ('duplicate required ID', ValueError, ['S1-A', 'S1-A'], example_truth,
     {'S1-A': {'S2-A'}}),
    ('extra prediction', KeyError, ['S1-A'], example_truth,
     {'S1-A': {'S2-A'}, 'S1-extra': set()}),
    ('missing truth', KeyError, ['S1-A'], {}, {'S1-A': {'S2-A'}}),
    ('empty evaluation', ValueError, [], {}, {}),
]
for label, error_type, required, truth, prediction in invalid_macro_cases:
    try:
        macro_score(required, truth, prediction)
    except error_type:
        pass
    else:
        raise AssertionError(f'{label} must fail validation')

# Verify the empty-prediction baseline without loading the full truth table.
truth_path = DATASET_DIR / 'train' / 'train_ground_truth.tsv'
truth_rows = empty_truth_rows = 0
for row in iter_tsv(truth_path):
    truth_rows += 1
    empty_truth_rows += not row['matched_entity_ids'].strip()
empty_baseline = empty_truth_rows / truth_rows
assert truth_rows == 2_206_821
assert empty_truth_rows == 123_247
assert isclose(empty_baseline, 123_247 / 2_206_821)
print(f'Macro example: {macro_score(example_truth, example_truth, example_predictions):.6f}')
print(f'Empty-prediction baseline: {empty_baseline:.8f}')
print('Macro checks pass: missing/extra predictions, duplicate references, missing truth, empty evaluation.')


Macro example: 0.333333
Empty-prediction baseline: 0.05584821
Macro checks pass: missing/extra predictions, duplicate references, missing truth, empty evaluation.


## 6. Inspect 20–30 complete labeled groups
Choose references across both countries, singletons, and several match counts. Obtain their target IDs from ground truth, then stream S2/S3 to collect only those targets. Avoid building a lookup dict of every source record.

The inspection cell below selects a deterministic convenience sample of 25 complete groups and 57 target records. It includes 13 US and 12 India references, with match counts 0, 1, 2, and 5; no ten-match group was selected. Nine target addresses are blank. The display now includes full names and addresses for each labeled pair, plus explicit singleton rows. Five useful patterns observed:

- US records often preserve a strong name/address core but add typos, spacing changes, punctuation, or legal-suffix variation.
- Indian records can appear in English, Devanagari, or another transliteration, so ASCII-only normalization would lose signal.
- One reference can have several true variants, including multiple S2 or S3 records; predicting only one match loses recall.
- Some matches have weak or unrelated-looking names, so address and country evidence must support the decision.
- Empty-label references are real singletons; adding a plausible-looking candidate creates a complete entity-level false positive.

Two examples worth inspecting side by side:

- `S1-777597828`: **Uptown Pub** matches **Solkeloquo** (`S3-413669121`); the address retains 6114, Tenth Avenue, and Spokane Valley despite the name change.
- `S1-473377609`: **Wave & Brothers Ltd** matches **Belobrixlum** (`S3-433876173`); both addresses retain 65, Mahada Colony, Sai Nagar, and Nagpur.

These are labeled training examples, not rules that address agreement alone always proves identity.


In [9]:
import pandas as pd

# Select the first 25 eligible complete groups: a deterministic convenience sample.
# This is not random, balanced, or guaranteed to include each allowed match count.
selected_truth = {}
for row in iter_tsv(DATASET_DIR / 'train' / 'train_ground_truth.tsv'):
    match_ids = parse_match_ids(row['matched_entity_ids'])
    match_count = len(match_ids)
    if match_count in {0, 1, 2, 5, 10}:
        selected_truth[row['source1_entity_id']] = match_ids
        if len(selected_truth) >= 25:
            break

selected_ids = set(selected_truth)
selected_source1 = {}
for row in iter_tsv(DATASET_DIR / 'train' / 'train_source1.tsv'):
    if row['entity_id'] in selected_ids:
        selected_source1[row['entity_id']] = row

selected_targets = set().union(*selected_truth.values())
selected_sources = {}
for source_number in (2, 3):
    path = DATASET_DIR / 'train' / f'train_source{source_number}.tsv'
    for row in iter_tsv(path):
        if row['entity_id'] in selected_targets:
            selected_sources[row['entity_id']] = row

# Fail explicitly if a sampled label references a record that was not found.
assert selected_ids == selected_source1.keys(), 'Sampled reference records are missing'
assert selected_targets == selected_sources.keys(), 'Sampled target records are missing'

inspection_rows = []
pair_rows = []
for reference_id, target_ids in selected_truth.items():
    reference = selected_source1[reference_id]
    inspection_rows.append({
        'reference_id': reference_id,
        'country': reference['country'],
        'match_count': len(target_ids),
        'reference_name': reference['business_name'],
    })
    # Preserve singletons as explicit inspection rows, not artificial match pairs.
    for target_id in sorted(target_ids) or [None]:
        target = selected_sources[target_id] if target_id is not None else None
        pair_rows.append({
            'reference_id': reference_id,
            'reference_country': reference['country'],
            'reference_name': reference['business_name'],
            'reference_address': reference['business_address'],
            'target_id': target_id if target is not None else '(singleton)',
            'target_country': target['country'] if target is not None else '',
            'target_name': target['business_name'] if target is not None else '',
            'target_address': target['business_address'] if target is not None else '',
            'target_address_missing': not target['business_address'].strip() if target is not None else None,
        })

labeled_groups = pd.DataFrame(inspection_rows)
labeled_pairs = pd.DataFrame(pair_rows)
print('Group summary:')
display(labeled_groups)
print('Full reference/target comparisons (blank target address is missing evidence):')
with pd.option_context('display.max_colwidth', None, 'display.max_columns', None,
                       'display.max_rows', None):
    display(labeled_pairs)
print(f'Inspected {len(labeled_groups)} complete groups and {len(selected_targets)} target records.')
print('Country coverage:', labeled_groups['country'].value_counts().to_dict())
print('Match-count coverage:', labeled_groups['match_count'].value_counts().sort_index().to_dict())
print('Targets with blank addresses:', sum(not r['business_address'].strip() for r in selected_sources.values()))


Group summary:
Full reference/target comparisons (blank target address is missing evidence):
Inspected 25 complete groups and 57 target records.
Country coverage: {'US': 13, 'India': 12}
Match-count coverage: {0: 4, 1: 3, 2: 12, 5: 6}
Targets with blank addresses: 9


,reference_id,country,match_count,reference_name
0,S1-965667,US,5,Maure Williams Colombier Inc
1,S1-318373630,India,2,Red Ventures Private Limited
2,S1-86989137,India,2,Laxmi Golden Investments Private Limited
3,S1-29845983,US,2,Hendricks and Flowers Inc
4,S1-730934468,US,2,Orellana Investments LLC
5,S1-692000596,US,2,"Desai, Alanah, Esq., PC"
6,S1-561341312,India,2,Balaji Investment Private Limited
7,S1-777597828,US,2,Uptown Pub
8,S1-282467635,India,2,Prem & Sons Pvt Ltd
9,S1-727602285,India,5,Swastik Om Solutions LLP


,reference_id,reference_country,reference_name,reference_address,target_id,target_country,target_name,target_address,target_address_missing
0,S1-965667,US,Maure Williams Colombier Inc,"85 Wayne Avenue, Ticonderoga, NY",S2-681193310,US,Maure Wilblims Colombier Inc,,True
1,S1-965667,US,Maure Williams Colombier Inc,"85 Wayne Avenue, Ticonderoga, NY",S2-743505751,US,Maure Williams Colombier,,True
2,S1-965667,US,Maure Williams Colombier Inc,"85 Wayne Avenue, Ticonderoga, NY",S3-11291185,US,maurewilliamscolombier.com,"Wayne Ave, Ticonderoga Townshiip, New York",False
3,S1-965667,US,Maure Williams Colombier Inc,"85 Wayne Avenue, Ticonderoga, NY",S3-775321672,US,Dréxkor,"85 Wanye Avenue, Ticonderoga Townshiip, New York",False
4,S1-965667,US,Maure Williams Colombier Inc,"85 Wayne Avenue, Ticonderoga, NY",S3-860443364,US,Maure Williams Inc Center,,True
5,S1-318373630,India,Red Ventures Private Limited,"Rajasthan, Jaipur, Banipark, Gokul Apartment, E-3A Kanti Chandra Road, G-1",S2-660036492,India,रेड वेंचर्स प्राइवेट लिमिटेड,"G-1, BANIPARK, JAIPUR, Rajasthan",False
6,S1-318373630,India,Red Ventures Private Limited,"Rajasthan, Jaipur, Banipark, Gokul Apartment, E-3A Kanti Chandra Road, G-1",S3-804600254,India,Red Ventures Private,"Doro No 316 G-1, Gokul Apartment, E-3a Kanti Chandra Road, Banipark, Subhash Nagar, RJ",False
7,S1-86989137,India,Laxmi Golden Investments Private Limited,"New Bridge Business Centre'S 11Th Floor, N1 Block Embassy Manyata Business Tech Park, Naga, Wara, Bangalore, Karnataka",S3-274817120,India,Laxmi Gbn lnvestments Private Limited,"New Bridge Bssiness Centre's 11Th Floor, N1 Block Embassy Manyata Business Tech Park, Naga, Wara, Bangalore, ಕರ್ನಾಟಕ",False
8,S1-86989137,India,Laxmi Golden Investments Private Limited,"New Bridge Business Centre'S 11Th Floor, N1 Block Embassy Manyata Business Tech Park, Naga, Wara, Bangalore, Karnataka",S3-312496301,India,Laxmi Golden Investments,"New Bridge Buisness Centre's 11Th Floor, N1 Block Embassy Manyata Business Tech Park, Naga, Wara, Bangalore, KA",False
9,S1-29845983,US,Hendricks and Flowers Inc,"33 Sleepy Hollow Drive, Danbury, CT",S2-648035184,US,Hendricks and Flowers Inc,"CT, SLEEPY HOLLOW DRIVE, DANBURY",False


## 7. Your validation plan
Write your reasoning before generating split files.

- What is the business group you will keep together?
- What if a target ID is linked to two references?
- How will you preserve countries and match-count buckets?
- Where will unmatched target distractors go?
- How will you exclude held-out identities from supervised training negatives?
- How will you save the seed and split manifest?
- Why does a US/India holdout not directly measure France performance?

Proposed starting proportions are 70/15/15; they have NOT been applied yet.

**Our split plan:**

Use connected labeled business groups as the split unit. Start by linking references that share a labeled target ID; keep every connected component together so a target variant cannot appear in both training and validation. Assign components to train/development/locked-holdout with seed `2026`, approximately 70/15/15, while preserving country and match-count buckets as closely as possible. Keep unmatched target records as distractors associated with their split, and never use held-out identities as supervised training negatives. Save the component assignments, seed, bucket counts, and dataset/configuration hashes in `data/processed/`. A US/India holdout measures in-domain generalization; France performance cannot be measured locally without labels. Training on US and evaluating India, and then reversing the direction, provides proxy country-transfer stress tests; these do not directly estimate France performance.


## Phase 1 complete

- Parser handles empty lists, duplicate IDs, malformed prefixes, empty tokens, and internal whitespace.
- Entity-level and macro $F_{0.5}$ checks pass, including the full empty-prediction baseline and strict required-reference coverage checks.
- Twenty-five labeled groups were inspected across countries and match-count buckets.
- The split design keeps connected business identities together; proxy country-transfer stress tests do not measure France performance directly.

Phase 2 should begin with a reproducible connected-component split manifest, followed by conservative normalization and an exact/normalized matching baseline. Do not use the file-head previews as training data, and do not create all-pairs comparisons.
